# 03 - Comparación de modelos

Carga los modelos entrenados desde `models/` y compara sus salidas. Inspecciona feature importance del XGBoost, scoreline grids del Poisson, y probabilidades blendeadas + calibradas.

**Prerequisito**: haber corrido `scripts/train_models.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.models.base_model import BaseOutcomeModel
from src.models.calibration import ProbabilityCalibrator
from src.models.poisson_model import PoissonScoreModel
from src.prediction.feature_builder import build_inference_feature_matrix, filter_upcoming
from src.prediction.predictor import MatchPredictor
from src.utils.io import load_pickle

MODELS_DIR = ROOT / 'models'

## Carga de los modelos entrenados

In [ ]:
xgb = BaseOutcomeModel.load(MODELS_DIR / 'xgboost_model.pkl')
mn = BaseOutcomeModel.load(MODELS_DIR / 'multinomial_model.pkl')
poisson = load_pickle(MODELS_DIR / 'poisson_model.pkl')
calibrator = load_pickle(MODELS_DIR / 'calibrator.pkl')

print(f'XGBoost      : {xgb.name}, fitted={xgb.is_fitted}, features={len(xgb.feature_columns)}')
print(f'Multinomial  : {mn.name}, fitted={mn.is_fitted}, features={len(mn.feature_columns)}')
print(f'Poisson      : fitted={poisson.is_fitted}, max_goals={poisson.max_goals}, rho={poisson.rho}')
print(f'Calibrator   : strategy={calibrator.strategy.value}, fitted={calibrator.is_fitted}')

## Feature importance del XGBoost (top 20)

In [ ]:
imp = xgb.feature_importance().head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp['feature'][::-1], imp['importance'][::-1], color='#2ca02c')
ax.set_xlabel('Importance')
ax.set_title('Top 20 features XGBoost')
plt.tight_layout()
plt.show()

print(imp.to_string(index=False))

## Coeficientes del Poisson (top equipos por ataque y defensa)

Attack > 1.0: mejor que la media. Defense > 1.0: peor que la media (concede más).

In [ ]:
fit = poisson.fit_result
attack_df = pd.DataFrame({'team': list(fit.attack.keys()), 'attack': list(fit.attack.values())})
defense_df = pd.DataFrame({'team': list(fit.defense.keys()), 'defense': list(fit.defense.values())})

print(f'Intercept (log avg goals): {fit.intercept:.3f}  -> avg ≈ {np.exp(fit.intercept):.2f} goles')
print(f'Home advantage: {fit.home_advantage:.3f}')
print()
print('Top 15 ataque:')
print(attack_df.sort_values('attack', ascending=False).head(15).to_string(index=False))
print()
print('Top 15 defensa (menor = mejor):')
print(defense_df.sort_values('defense', ascending=True).head(15).to_string(index=False))

## Comparación de probabilidades por modelo en un partido de muestra

Elegimos dos equipos top del Mundial 2026 y vemos qué dice cada modelo.

In [ ]:
TEAM_A = 'Brazil'
TEAM_B = 'Argentina'

feature_matrix = build_inference_feature_matrix(tournament_year=2026)
row = feature_matrix[(feature_matrix['team_a'] == TEAM_A) & (feature_matrix['team_b'] == TEAM_B)]
if row.empty:
    # invertido
    row = feature_matrix[(feature_matrix['team_a'] == TEAM_B) & (feature_matrix['team_b'] == TEAM_A)]
if row.empty:
    # buscamos cualquier partido de Brasil para tomar las features de Brasil como team_a
    row = feature_matrix[feature_matrix['team_a'] == TEAM_A].head(1)
X = row[xgb.feature_columns]
print(f'Match: {TEAM_A} vs {TEAM_B}')
print()

p_xgb = xgb.predict_proba(X)[0]
p_mn = mn.predict_proba(X)[0]
p_poisson = poisson.outcome_probabilities(TEAM_A, TEAM_B)

print(f'XGBoost      : H={p_xgb[0]:.3f}  D={p_xgb[1]:.3f}  A={p_xgb[2]:.3f}')
print(f'Multinomial  : H={p_mn[0]:.3f}  D={p_mn[1]:.3f}  A={p_mn[2]:.3f}')
print(f'Poisson      : H={p_poisson[0]:.3f}  D={p_poisson[1]:.3f}  A={p_poisson[2]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
outcomes = ['Home', 'Draw', 'Away']
x = np.arange(3)
w = 0.25
ax.bar(x - w, p_xgb, w, label='XGBoost', color='#1f77b4')
ax.bar(x, p_mn, w, label='Multinomial', color='#ff7f0e')
ax.bar(x + w, p_poisson, w, label='Poisson', color='#2ca02c')
ax.set_xticks(x)
ax.set_xticklabels(outcomes)
ax.set_ylabel('Probabilidad')
ax.set_title(f'{TEAM_A} vs {TEAM_B}: comparación por modelo')
ax.legend()
plt.tight_layout()
plt.show()

## Scoreline grid Poisson (Dixon-Coles ajustado)

In [ ]:
grid = poisson.score_matrix(TEAM_A, TEAM_B)
max_g = min(grid.shape[0], 6)
grid = grid[:max_g, :max_g]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(grid, cmap='YlOrRd', origin='lower')
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        ax.text(j, i, f'{grid[i, j]:.3f}', ha='center', va='center', fontsize=8)
ax.set_xlabel(f'Goles {TEAM_B}')
ax.set_ylabel(f'Goles {TEAM_A}')
ax.set_title(f'Scoreline grid: {TEAM_A} vs {TEAM_B}')
fig.colorbar(im)
plt.tight_layout()
plt.show()

print('Top 5 scorelines más probables:')
for (sa, sb), p in poisson.top_k_scorelines(TEAM_A, TEAM_B, k=5):
    print(f'  {sa}-{sb}: {p:.4f}')

## Predicción ensemble + calibrada para todas las fixtures de grupos del Mundial 2026

In [ ]:
predictor = MatchPredictor(
    outcome_models={'multinomial': mn, 'xgboost': xgb},
    poisson_model=poisson,
    calibrator=calibrator,
)
fixtures = filter_upcoming(feature_matrix, stage_substr='group')
print(f'Fixtures upcoming de grupos: {len(fixtures)}')

preds = predictor.predict_matches(fixtures)
print()
print('Primeros 15 partidos predichos:')
print(preds.head(15).to_string(index=False))

## Distribución de confianza máxima

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(preds['confidence'], bins=30, color='#1f77b4', edgecolor='black')
ax.set_xlabel('Confianza del pick argmax')
ax.set_ylabel('Cantidad de fixtures')
ax.set_title('Distribución de confianza en las predicciones del Mundial 2026')
ax.axvline(preds['confidence'].mean(), color='red', linestyle='--', label=f'Media={preds["confidence"].mean():.2f}')
ax.legend()
plt.tight_layout()
plt.show()

by_outcome = preds['most_likely_outcome'].value_counts()
print('Distribución de picks argmax:')
print(by_outcome.to_string())